# Character-conditioned handwriting generation — Graves LSTM-MDN
Based on Alex Graves, *Generating Sequences With Recurrent Neural Networks*, arXiv:1308.0850v5, §§2, 4.1–4.3, 5.4 ([paper](https://arxiv.org/abs/1308.0850)).

Uses your existing UJI preprocessing pickle, writer split and three pen states. Implements correlated bivariate Gaussian mixtures, teacher forcing, masked likelihood, skip-connected peephole LSTM, centered RMSprop and biased/unbiased autoregressive sampling. Unlike the paper's line-level soft-window text conditioning, a known single character is embedded at every step. Adaptive weight noise is not implemented. Default model is smaller than the paper; change configuration for larger experiments. No original files are modified.

In [ ]:
#imports
import matplotlib.pyplot as plt
import numpy as np
import pickle

import copy
from pathlib import Path


import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

# Set random seeds for reproducibility (same as Exercise 2)
np.random.seed(42)
torch.manual_seed(42)

print("PyTorch Version:", torch.__version__)
import math
import torch.nn.functional as F


In [ ]:
# Use the same dataset as your deterministic baseline for a fair comparison.
DATA_PATH = Path("delta_data_equal_length.pkl")
# Alternative: Path("delta_data_normalized.pkl") for variable-length sequences.
with DATA_PATH.open("rb") as file:
    delta_data = pickle.load(file)


In [ ]:
def deltas_to_strokes(trajectory):
    """
    Reconstructs (x, y) strokes from (dx, dy, pen_state).
    
    pen_state = 0: pen is DOWN (this step is part of a stroke -> draw it)
    pen_state = 1: pen is UP   (this step is an air move -> skip it)
    pen_state = 2: END         (stop before applying this step)
    """
    strokes = []
    current_stroke = []
    
    x, y = 0.0, 0.0
    
    for dx, dy, pen_state in trajectory:
        if pen_state == 2:
            break

        prev_x, prev_y = x, y
        x = x + dx
        y = y + dy
        
        if pen_state == 0:
            # Pen is DOWN: include this step in the current stroke
            if not current_stroke:
                # First point of the stroke: include the "from" position too
                current_stroke.append((prev_x, prev_y))
            current_stroke.append((x, y))
            
        else:
            # Pen is UP: flying through the air
            # Save the stroke we were drawing (if any) and reset
            if current_stroke:
                strokes.append(current_stroke)
                current_stroke = []
                
    # Save the last stroke if it exists
    if current_stroke:
        strokes.append(current_stroke)        
    return strokes




In [ ]:
train_data = {}
val_data = {}
test_data = {}

# 1. Collect all test writer IDs
test_writer_ids = []
for writer_id in delta_data:
    if writer_id.startswith("tst_"):
        test_writer_ids.append(writer_id)

test_writer_ids.sort()

# 2. Split the 20 test writers in half (10 for val, 10 for test)
val_writers = test_writer_ids[:10]
test_writers = test_writer_ids[10:]

# 3. Fill the dictionaries
for writer_id in delta_data:
    if writer_id.startswith("trn_"):
        train_data[writer_id] = delta_data[writer_id]
    elif writer_id in val_writers:
        val_data[writer_id] = delta_data[writer_id]
    elif writer_id in test_writers:
        test_data[writer_id] = delta_data[writer_id]

# 4. Count total samples in each set
def count_samples(dataset):
    total = 0
    for writer_id in dataset:
        for char in dataset[writer_id]:
            total += len(dataset[writer_id][char])
    return total

print(f"Training set:   {len(train_data)} writers | {count_samples(train_data)} samples")
print(f"Validation set: {len(val_data)} writers | {count_samples(val_data)} samples")
print(f"Test set:       {len(test_data)} writers | {count_samples(test_data)} samples")

## Teacher forcing and masks
Remove the initial artificial start token stored by Project.ipynb from the targets (but retain it as the first input). Keep the terminal token for pen classification. Fit coordinate normalization using training motion points only. The input and output coordinates are in normalized model space; generated displacements are transformed back to the pickle's coordinate space.

In [ ]:
char_to_id = {c:i for i,c in enumerate(sorted({c for chars in train_data.values() for c in chars}))}
num_chars = len(char_to_id)
START = np.array([0.,0.,1.], dtype=np.float32)
def clean(rep):
    a = np.asarray(rep, dtype=np.float32)
    assert a.ndim == 2 and a.shape[1] == 3 and np.isfinite(a).all()
    if len(a) and np.allclose(a[0], START): a = a[1:]
    assert len(a) and a[-1,2] == 2 and np.all(np.isin(a[:,2], [0,1,2]))
    return a.copy()
train_xy = np.concatenate([a[a[:,2]!=2,:2] for chars in train_data.values() for reps in chars.values() for rep in reps for a in [clean(rep)]])
coord_mean = train_xy.mean(0)
coord_std = np.maximum(train_xy.std(0), 1e-6)
def build_dataset_lists(split):
    ids, X, Y = [], [], []
    for chars in split.values():
        for char,reps in chars.items():
            for rep in reps:
                a = clean(rep)
                a[:,:2] = (a[:,:2]-coord_mean)/coord_std
                # EOS has no physical displacement; its coordinate loss is masked.
                a[a[:,2]==2,:2] = 0
                start = START.copy(); start[:2] = -coord_mean/coord_std
                ids.append(char_to_id[char]); X.append(np.vstack([start,a[:-1]])); Y.append(a)
    return ids,X,Y
c_train,X_train,y_train = build_dataset_lists(train_data)
c_val,X_val,y_val = build_dataset_lists(val_data)
c_test,X_test,y_test = build_dataset_lists(test_data)


In [ ]:
# 1. Find the maximum sequence length across all sets
max_len = 0
for seq in X_train + X_val + X_test:
    if len(seq) > max_len:
        max_len = len(seq)

print("Maximum sequence length:", max_len)


# 2. Simple padding function
def pad_data(c_list, X_list, y_list, max_len):
    total_samples = len(X_list)

    # Create empty arrays filled with zeros
    X_padded = np.zeros((total_samples, max_len, 3), dtype=np.float32)
    y_padded = np.zeros((total_samples, max_len, 3), dtype=np.float32)
    mask = np.zeros((total_samples, max_len), dtype=bool)

    # Fill in the real data
    for i in range(total_samples):
        seq_len = len(X_list[i])

        # Copy original points
        X_padded[i, :seq_len, :] = X_list[i]
        y_padded[i, :seq_len, :] = y_list[i]

        # True for real points, False for padded zeros
        mask[i, :seq_len] = True

    c_array = np.array(c_list, dtype=np.int64)

    return c_array, X_padded, y_padded, mask


# 3. Apply padding to Train, Val, and Test
c_train_pad, X_train_pad, y_train_pad, mask_train = pad_data(c_train, X_train, y_train, max_len)
c_val_pad,   X_val_pad,   y_val_pad,   mask_val   = pad_data(c_val,   X_val,   y_val,   max_len)
c_test_pad,  X_test_pad,  y_test_pad,  mask_test  = pad_data(c_test,  X_test,  y_test,  max_len)

print("Padded X_train shape:", X_train_pad.shape)
print("Mask train shape:    ", mask_train.shape)

In [ ]:
# Convert to PyTorch Datasets
train_dataset = TensorDataset(
    torch.tensor(c_train_pad),
    torch.tensor(X_train_pad),
    torch.tensor(y_train_pad),
    torch.tensor(mask_train)
)

val_dataset = TensorDataset(
    torch.tensor(c_val_pad),
    torch.tensor(X_val_pad),
    torch.tensor(y_val_pad),
    torch.tensor(mask_val)
)

test_dataset = TensorDataset(
    torch.tensor(c_test_pad),
    torch.tensor(X_test_pad),
    torch.tensor(y_test_pad),
    torch.tensor(mask_test)
)


## Graves architecture (Sections 2 and 4)
Each recurrent layer receives the original input and the previous layer's output. All hidden layers feed the prediction head. Peephole connections link cell states to the gates. The output has 6M + 3 parameters: M mixture logits, 2M means, 2M log scales, M correlations and three pen logits. The three-class categorical head replaces Graves's binary end-of-stroke output.

In [ ]:
def clip_activation_grad(x, limit):
    if x.requires_grad: x.register_hook(lambda g: g.clamp(-limit, limit))
    return x

class PeepholeLSTM(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.input = nn.Linear(input_size, 4*hidden_size)
        self.recurrent = nn.Linear(hidden_size, 4*hidden_size, bias=False)
        self.pi = nn.Parameter(torch.zeros(hidden_size))
        self.pf = nn.Parameter(torch.zeros(hidden_size))
        self.po = nn.Parameter(torch.zeros(hidden_size))
        with torch.no_grad(): self.input.bias[hidden_size:2*hidden_size].fill_(1.)
    def forward(self, x, state=None):
        B,T,_ = x.shape
        h,c = state if state is not None else (x.new_zeros(B,self.hidden_size),x.new_zeros(B,self.hidden_size))
        inputs = self.input(x)
        outputs = []
        for t in range(T):
            i,f,g,o = (inputs[:,t]+self.recurrent(h)).chunk(4,-1)
            i = torch.sigmoid(clip_activation_grad(i+self.pi*c,10))
            f = torch.sigmoid(clip_activation_grad(f+self.pf*c,10))
            g = torch.tanh(clip_activation_grad(g,10))
            c = f*c+i*g
            o = torch.sigmoid(clip_activation_grad(o+self.po*c,10))
            h = o*torch.tanh(c)
            outputs.append(h)
        return torch.stack(outputs,1),(h,c)

class HandwritingLSTM(nn.Module):
    def __init__(self,num_chars,hidden_size=32,num_layers=4,embedding_dim=16,mixtures=20,dropout_p=0.):
        super().__init__()
        self.mixtures = mixtures
        self.char_embedding = nn.Embedding(num_chars,embedding_dim)
        d = 3+embedding_dim
        self.layers = nn.ModuleList([PeepholeLSTM(d+(hidden_size if i else 0),hidden_size) for i in range(num_layers)])
        self.dropout = nn.Dropout(dropout_p)
        self.head = nn.Linear(hidden_size*num_layers,6*mixtures+3)
    def forward(self,x,c,hidden=None):
        base = torch.cat([x,self.char_embedding(c)[:,None,:].expand(-1,x.size(1),-1)],-1)
        outputs,states = [],[]
        for i,layer in enumerate(self.layers):
            inp = base if i==0 else torch.cat([base,self.dropout(outputs[-1])],-1)
            h,state = layer(inp,None if hidden is None else hidden[i])
            outputs.append(h); states.append(state)
        raw = clip_activation_grad(self.head(self.dropout(torch.cat(outputs,-1))),100)
        return raw,states

def unpack(raw,M):
    logits,mux,muy,lsx,lsy,rho = raw[...,:6*M].chunk(6,-1)
    # Numerical safeguards: bound log-scales and keep correlation away from +/-1.
    return logits,mux,muy,lsx.clamp(-5,5),lsy.clamp(-5,5),0.999*torch.tanh(rho),raw[...,6*M:]


## Negative log-likelihood (Graves Eqs. 23–26)
Compute the correlated Gaussian log density and combine mixture components with logsumexp. The coordinate likelihood excludes EOS and padding; pen cross entropy includes EOS but excludes padding. Default unweighted pen loss implements the categorical analogue of Graves's joint likelihood. Optional class weighting is provided for your pen-up imbalance experiment.

In [ ]:
def loss_terms(raw,target,mask,M,pen_weights=None):
    logits,mx,my,lx,ly,rho,pen = unpack(raw,M)
    dx = (target[...,0,None]-mx)*torch.exp(-lx)
    dy = (target[...,1,None]-my)*torch.exp(-ly)
    v = 1-rho.square()
    # Equivalent to (dx²+dy²-2*rho*dx*dy)/v, with less cancellation.
    z = dx.square()+(dy-rho*dx).square()/v
    log_pdf = -math.log(2*math.pi)-lx-ly-0.5*torch.log(v)-0.5*z
    nll = -torch.logsumexp(F.log_softmax(logits,-1)+log_pdf,-1)
    p = target[...,2].long()
    motion = mask & (p!=2)
    ce = F.cross_entropy(pen[mask],p[mask],weight=pen_weights,reduction='sum')
    return nll[motion].sum(),ce,motion.sum(),mask.sum()


In [ ]:
# Configuration: compact model comparable to your baseline.
# Paper's larger architecture: hidden_size=400, num_layers=3, mixtures=20.
MODEL_CONFIG = dict(num_chars=num_chars,hidden_size=32,num_layers=4,embedding_dim=16,mixtures=20,dropout_p=0.)
BATCH_SIZE = 128
EPOCHS = 100
PATIENCE = 15
PEN_WEIGHT = 1.0
USE_CLASS_WEIGHTS = False
LR = 1e-4
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CHECKPOINT = Path('handwriting_graves_mdn_best.pt')
train_loader = DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True)
val_loader = DataLoader(val_dataset,batch_size=BATCH_SIZE,shuffle=False)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE,shuffle=False)
model = HandwritingLSTM(**MODEL_CONFIG).to(DEVICE)
counts = torch.bincount(train_dataset.tensors[2][...,2][train_dataset.tensors[3]].long(),minlength=3).float()
pen_weights = (counts.max()/counts.clamp_min(1)).sqrt().to(DEVICE) if USE_CLASS_WEIGHTS else None
print(DEVICE, sum(p.numel() for p in model.parameters()), 'parameters; pen counts:',counts.tolist())
# Graves Eq. 38–41: epsilon inside square root, centered running variance.
class GravesRMSprop(torch.optim.Optimizer):
    def __init__(self,params,lr=1e-4,alpha=.95,momentum=.9,eps=1e-4):
        super().__init__(params,dict(lr=lr,alpha=alpha,momentum=momentum,eps=eps))
    @torch.no_grad()
    def step(self):
        for group in self.param_groups:
            a,m,lr,eps = (group[k] for k in ('alpha','momentum','lr','eps'))
            for p in group['params']:
                if p.grad is None: continue
                g = p.grad; s = self.state[p]
                if not s:
                    for k in ('n','g','delta'): s[k]=torch.zeros_like(p)
                s['n'].mul_(a).addcmul_(g,g,value=1-a)
                s['g'].mul_(a).add_(g,alpha=1-a)
                den = (s['n']-s['g'].square()).clamp_min(0).add(eps).sqrt()
                s['delta'].mul_(m).addcdiv_(g,den,value=-lr)
                p.add_(s['delta'])
optimizer = GravesRMSprop(model.parameters(),lr=LR)


In [ ]:
def run_epoch(loader,training=False):
    model.train(training)
    sums = np.zeros(4); conf = torch.zeros(3,3,dtype=torch.long)
    sqerr = 0.
    for c,x,y,mask in loader:
        c,x,y,mask = [z.to(DEVICE) for z in (c,x,y,mask)]
        # Trim batch padding to its actual maximum length.
        T = int(mask.sum(1).max()); x,y,mask = x[:,:T],y[:,:T],mask[:,:T]
        with torch.set_grad_enabled(training):
            raw,_ = model(x,c)
            a,b,nm,nv = loss_terms(raw,y,mask,model.mixtures,pen_weights)
            loss = (a+PEN_WEIGHT*b)/nv.clamp_min(1)
            if not torch.isfinite(loss): raise FloatingPointError('Nonfinite loss')
            if training:
                optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step()
        sums += [a.item(),b.item(),nm.item(),nv.item()]
        with torch.no_grad():
            l,mx,my,_,_,_,pen = unpack(raw,model.mixtures)
            pred = pen.argmax(-1); actual = y[...,2].long()
            conf += torch.bincount((3*actual[mask]+pred[mask]).cpu(),minlength=9).reshape(3,3)
            mean = torch.stack([(l.softmax(-1)*mx).sum(-1),(l.softmax(-1)*my).sum(-1)],-1)
            motion = mask & (actual!=2)
            sqerr += (((mean-y[...,:2])*torch.as_tensor(coord_std,device=DEVICE)).square().sum(-1)[motion]).sum().item()
    a,b,nm,nv=sums
    return dict(loss=(a+PEN_WEIGHT*b)/nv,coord_nll=a/max(nm,1),pen_loss=b/nv,
                delta_rmse=np.sqrt(sqerr/(2*max(nm,1))),pen_accuracy=conf.diag().sum().item()/nv,
                pen_up_recall=conf[1,1].item()/max(conf[1].sum().item(),1),eos_recall=conf[2,2].item()/max(conf[2].sum().item(),1))


In [ ]:
# Training and validation; keep the independent test set untouched.
history = {'train':[],'val':[]}
best = float('inf'); stale = 0
for epoch in range(1,EPOCHS+1):
    tr = run_epoch(train_loader,True); va = run_epoch(val_loader)
    history['train'].append(tr); history['val'].append(va)
    print(f"Epoch {epoch:03d} | train {tr['loss']:.4f} | val {va['loss']:.4f} | pen-up recall {va['pen_up_recall']:.3f} | EOS recall {va['eos_recall']:.3f}")
    if va['loss'] < best-1e-4:
        best=va['loss']; stale=0
        torch.save(dict(model_state=model.state_dict(),config=MODEL_CONFIG,char_to_id=char_to_id,coord_mean=coord_mean.tolist(),coord_std=coord_std.tolist(),epoch=epoch,history=history),CHECKPOINT)
    else:
        stale+=1
        if stale>=PATIENCE: break


In [ ]:
# Reload best checkpoint (can also be run after skipping the training cell).
ckpt = torch.load(CHECKPOINT,map_location=DEVICE,weights_only=True)
assert ckpt['char_to_id']==char_to_id, 'Character mapping differs from checkpoint'
assert np.allclose(ckpt['coord_mean'],coord_mean) and np.allclose(ckpt['coord_std'],coord_std), 'Dataset normalization differs'
model=HandwritingLSTM(**ckpt['config']).to(DEVICE)
model.load_state_dict(ckpt['model_state']); model.eval()
print('Best epoch:',ckpt['epoch'])
print('Held-out teacher-forced test metrics:',run_epoch(test_loader))


In [ ]:
fig,axes=plt.subplots(1,3,figsize=(13,4))
for ax,key in zip(axes,['loss','coord_nll','pen_up_recall']):
    for split in ('train','val'):
        ax.plot([r[key] for r in ckpt['history'][split]],label=split)
    ax.set_title(key); ax.set_xlabel('Epoch'); ax.legend(); ax.grid(alpha=.2)
plt.tight_layout(); plt.show()


## Autoregressive sampling (Section 5.4)
Bias b=0 gives unbiased samples from the fitted distribution; b>0 increases mixture-logit contrast and reduces Gaussian standard deviations by exp(-b). Correlated noise is generated using two independent standard normal draws. Pen states are sampled categorically by default; optional argmax is a robot-oriented decoding modification. The maximum step limit is a safety cap, not a learned termination.

In [ ]:
@torch.inference_mode()
def generate_trajectory(model,char,char_to_id,max_steps=200,bias=0.,pen_mode='sample'):
    if bias<0 or max_steps<1 or pen_mode not in ('sample','argmax'): raise ValueError('Invalid decoding options')
    p=next(model.parameters()); device=p.device
    c=torch.tensor([char_to_id[char]],device=device)
    mean=p.new_tensor(coord_mean); std=p.new_tensor(coord_std)
    x=torch.cat([-mean/std,p.new_tensor([1.])]).view(1,1,3)
    hidden=None; steps=[]; model.eval()
    for _ in range(max_steps):
        raw,hidden=model(x,c,hidden)
        l,mx,my,lx,ly,rho,pen=unpack(raw[0,-1],model.mixtures)
        s=pen.argmax().item() if pen_mode=='argmax' else torch.distributions.Categorical(logits=pen).sample().item()
        if s==2:
            steps.append([0.,0.,2.]); break
        j=torch.distributions.Categorical(logits=l*(1+bias)).sample()
        z1,z2=torch.randn(2,device=device)
        d=torch.stack([mx[j]+torch.exp(lx[j]-bias)*z1,my[j]+torch.exp(ly[j]-bias)*(rho[j]*z1+torch.sqrt(1-rho[j]**2)*z2)])
        physical=d*std+mean
        steps.append([*physical.cpu().tolist(),s])
        x=torch.cat([d,p.new_tensor([float(s)])]).view(1,1,3)
    return np.asarray(steps,dtype=np.float32)


In [ ]:
sample_chars=[c for c in ['a','b','c','A','B','z','2','u','8'] if c in char_to_id]
fig,axes=plt.subplots(len(sample_chars),3,figsize=(8,2*len(sample_chars)),squeeze=False)
for row,char in enumerate(sample_chars):
    for col,bias in enumerate([0.,0.5,1.]):
        ax=axes[row,col]; traj=generate_trajectory(model,char,char_to_id,max_steps=max(200,2*max_len),bias=bias)
        for stroke in deltas_to_strokes(traj):
            xy=np.asarray(stroke); ax.plot(xy[:,0],xy[:,1],'-o',markersize=1.5)
        ax.set_aspect('equal'); ax.set_title(f'{char!r}, b={bias} | '+('EOS' if traj[-1,2]==2 else 'capped')); ax.axis('off')
plt.tight_layout(); plt.show()


In [ ]:
# Generate every character as in your original notebook.
generated_chars=sorted(char_to_id)
ncols=8; nrows=math.ceil(len(generated_chars)/ncols)
fig,axes=plt.subplots(nrows,ncols,figsize=(2.5*ncols,2.5*nrows),squeeze=False)
for ax,char in zip(axes.flat,generated_chars):
    traj=generate_trajectory(model,char,char_to_id,max_steps=max(200,2*max_len),bias=0.75)
    for stroke in deltas_to_strokes(traj):
        xy=np.asarray(stroke); ax.plot(xy[:,0],xy[:,1],'-o',markersize=1.5)
    ax.set_title(f'{char!r}'+('' if traj[-1,2]==2 else ' (capped)')); ax.set_aspect('equal'); ax.axis('off')
for ax in axes.flat[len(generated_chars):]: ax.axis('off')
plt.tight_layout(); plt.show()


## Interpretation
NLL may be negative because continuous probability densities may exceed one. Delta RMSE here is teacher-forced displacement RMSE in pickle coordinates, not reconstructed trajectory RMSE. Compare free-running samples, termination, DTW and smoothness separately. For an isolated output-head ablation, also keep your original stacked LSTM architecture and optimizer fixed; this notebook intentionally changes those to follow Graves more closely. The custom peephole recurrence is slower than fused PyTorch LSTM, especially on GPU.